In [0]:
#Etapa bronze: aca se lee el archivo crudo montado en Catalog / Schema / Volumen
print ('Etapa bronze')
#leemos el archivo, format le indica el formato, header que tiene la fila de cabecera, inferSchema para que infiera el tipo de dato y load para indicar el path
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("delimiter", ";") \
    .load("/Volumes/workspace/default/misarchivos/info.csv")
#muestra lo que se leyo en el data frame
display(df)

In [0]:
from pyspark.sql.functions import col, trim, upper

#Etapa silver: aca se toma el data frame de la etapa anterior y se separan los campos, se limpian los campos y se guarda en el data frame df_clean
print ('Etapa silver')

#dropna elimina los registros que tengan null en cualquier columna y dropDuplicates elimina los duplicados, si queremos que elimine solo los nulos en una columna dropna(subset=["edad"]) si queremos los que tengan todo en null df.dropna(how="all")
df_clean = df \
    .withColumn("Nombre", upper(trim(col("Nombre")))) \
    .withColumn("Apellido", upper(trim(col("Apellido")))) \
    .withColumn("Edad",trim(col("Edad"))) \
    .dropna()

display(df_clean)

df_clean.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.alumnos_silver")

In [0]:
%sql
SELECT 
    avg(Edad) AS promedio_edad
FROM workspace.default.alumnos_silver;


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.alumnos_gold AS
SELECT concat_ws(' ',Nombre,Apellido) as NombreCompleto, Edad
FROM workspace.default.alumnos_silver;

SELECT * FROM workspace.default.alumnos_gold;

In [0]:
df_clean.write.format("parquet") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/misarchivos/clientes_parquet")

df_parquet = spark.read.parquet("/Volumes/workspace/default/misarchivos/clientes_parquet")
display(df_parquet)

In [0]:
data = [("Juan", 30), ("Ana", 25)]
df = spark.createDataFrame(data, ["Nombre", "Edad"])
display(df)